# पाठ ०१ - एआई एजेन्टहरूमा परिचय

**शुरुआतीहरूका लागि एआई एजेन्टहरू** कोर्सको पहिलो पाठमा स्वागत छ!

एक **एआई एजेन्ट** यस्तो कार्यक्रम हो जसले reasoning इन्जिनको रूपमा ठूलो भाषा मोडेल (LLM) प्रयोग गर्छ र वास्तविक संसारमा *कार्यहरू* गर्न सक्छ — API हरू कल गर्ने, डाटाबेस सोध्ने, वा कोड चलाउने — प्रयोगकर्ताको तर्फबाट लक्ष्य प्राप्त गर्न।

यस नोटबुकमा तपाईं आफ्नो पहिलो एजेन्ट बनाउनेछौं: एक **ट्राभल एजेन्ट** जसले अवकाश गन्तव्यहरू सिफारिस गर्छ। यस क्रममा तपाईं सिक्नुहुनेछ कसरी:

१. **Microsoft Agent Framework** प्रयोग गरेर Microsoft Foundry Agent Service मा जडान गर्ने।
२. एजेन्टलाई एक **टूल** दिने — एउटा साधारण Python function जसलाई एजेन्ट कल गर्न सक्छ।
३. एजेन्ट चलाउने र यसको प्रतिक्रिया निरीक्षण गर्ने।
४. एजेन्टको प्रतिक्रियालाई टोकन-टोकन अनुसार स्ट्रिम गर्ने।


## सेटअप

यो नोटबुक चलाउनु अघि, सुनिश्चित गर्नुहोस् कि तपाईंसँग छ:

1. **एक Microsoft Foundry परियोजना** जसमा एक तैनाथ गरिएको च्याट मोडेल छ (जस्तै `gpt-4.1-mini`)।
2. **Azure CLI मा लगइन गरिएको छ** — आफ्नो टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण चरहरू सेट गरिएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Microsoft Foundry परियोजना अन्तबिन्दु।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंको तैनाथ गरिएको मोडेलको नाम।

तलको सेल्लले तपाईंलाई आवश्यक Python प्याकेजहरू स्थापना गर्दछ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## तपाईंको पहिलो एजेन्ट सिर्जना गर्दै

एउटा एजेन्टलाई दुई कुरा चाहिन्छ:

- **निर्देशहरू** जसले यसलाई *को हो* र *कसरी व्यवहार गर्ने* (सिस्टम प्रॉम्प्ट) बताउँछ।
- **उपकरणहरू** — Python कार्यहरू जुन `@tool` द्वारा सजाइएको हुन्छन् जसलाई एजेन्टले जानकारी प्राप्त गर्न वा कार्यहरू गर्न कल गर्न सक्छ।

तल हामी एउटा सरल उपकरण परिभाषित गर्छौं जुन लोकप्रिय भ्रमण गन्तव्यहरूको सूची फिर्ता गर्छ। जब प्रयोगकर्ताले यात्रा सिफारिसहरू सोध्छ, एजेन्टले यो उपकरण प्रयोग गर्नेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिङ प्रतिक्रिया

अझ अन्तरक्रियात्मक अनुभवको लागि तपाईं एजेन्टको प्रतिक्रियालाई **स्ट्रीम** गर्न सक्नुहुन्छ। पूर्ण जवाफको पर्खाइ नगरी, एजेन्टले जति टेक्स्ट खण्डहरू उत्पन्न गर्छ, त्यति नै प्रस्तुत गर्दछ। यो विशेष गरी च्याट इन्टरफेसहरूमा उपयोगी हुन्छ जहाँ तपाईंले वास्तविक समयमा आउटपुट देखाउन चाहनुहुन्छ।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

- **प्रदाता सिर्जना गर्ने** जसले Microsoft Foundry Agent Service सँग `FoundryChatClient` मार्फत जडान गर्छ।
- **उपकरण परिभाषित गर्ने** `@tool` डेकोरेटर प्रयोग गरेर ताकि एजेन्टले तपाईंका Python कार्यहरू कल गर्न सकोस्।
- **एजेन्ट चलाउने** प्रयोगकर्ता सन्देशसँग र यसको प्रतिक्रिया प्रिन्ट गर्ने।
- **प्रतिक्रियाहरू स्ट्रिम गर्ने** तत्काल परिणामको लागि।

अर्को पाठमा हामी एजेन्टिक फ्रेमवर्कहरूलाई अझ गहिरो रूपमा अन्वेषण गर्नेछौं र एजेन्टहरूलाई अझ शक्तिशाली उपकरणहरू र बहु-चरण तर्क क्षमताहरू कसरी दिन सक्छौं भनेर सिक्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
